# SD Sketch Coach — Colab Experiment v0

이 노트북은 **GitHub의 동일한 ML 코드**를 Colab에서 실행합니다.

순서: `clone → install → data check → geometry baseline → MLP → evaluation → ONNX export/verify`.

In [ ]:
# 1) 자신의 GitHub 저장소 URL로 변경하세요.
GITHUB_REPO_URL = "https://github.com/YOUR_ID/sd-sketch-coach.git"
REPO_DIR = "sd-sketch-coach"

In [ ]:
# 2) 저장소 clone
import os, shutil
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone {GITHUB_REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# 3) 의존성 설치
!pip -q install -r requirements.txt

In [ ]:
# 4) 데이터 확인
import json
from collections import Counter

rows=[]
with open("data/synthetic_v0.jsonl", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

print("samples:", len(rows))
print(Counter(r["label"] for r in rows))
print("example features:", rows[0]["features"])

In [ ]:
# 5) Geometry baseline
!python ml/geometry_baseline.py

In [ ]:
# 6) MLP 학습
!python ml/train_mlp.py --epochs 120

In [ ]:
# 7) held-out synthetic TEST split 평가
# 주의: 여전히 synthetic-domain sanity check이며 real-user generalization 성능이 아닙니다.
!python ml/evaluate.py

In [ ]:
# 8) ONNX export
!python ml/export_onnx.py

In [ ]:
# 9) PyTorch ↔ ONNX 출력 검증
!python ml/verify_onnx.py

In [ ]:
# 10) 산출물 확인
from pathlib import Path
for p in Path("models").glob("*"):
    if p.is_file():
        print(p, f"{p.stat().st_size/1024:.1f} KB")

## 다음 실험

V0가 정상 동작하면 다음 순서로 확장합니다.

1. train/val/test용 **template variation** 확대
2. 합성 오류의 범위를 더 현실적으로 조정
3. real beginner drawing 30~50개로 **domain gap 측정**
4. raw `(x,y,t)`를 사용하는 1D-CNN baseline 추가
5. ONNX 모델을 React Native + iPhone 14 Pro에서 benchmark